# AI Carbon Emission Prediction & Monitoring - ML Training Notebook
This notebook loads the Tamil Nadu carbon emission dataset, performs exploratory data analysis, encodes categorical variables, scales features, compares 4 regression algorithms (Linear Regression, Random Forest, Gradient Boosting, Extra Trees), evaluates performance metrics (MAE, MSE, RMSE, R²), and exports the optimal model artifacts for Flask application deployment.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load Dataset
df = pd.read_csv('../dataset/carbon_emission_dataset.csv')
df.head()

In [2]:
# Data Summary
print('Dataset Shape:', df.shape)
df.info()
df.describe()

In [3]:
# Categorical Encoding & Feature Scaling
X = df.drop(columns=['Carbon_Emission_kgCO2'])
y = df['Carbon_Emission_kgCO2']

label_encoders = {}
X_encoded = X.copy()
for col in ['District', 'Place', 'Area_Type']:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X[col])
    label_encoders[col] = le

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_encoded)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [4]:
# Model Training & Evaluation
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, preds)
    results.append({'Model': name, 'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2})

results_df = pd.DataFrame(results)
print(results_df.sort_values(by='R2', ascending=False))

In [5]:
# Save Best Model
best_model = RandomForestRegressor(n_estimators=100, random_state=42)
best_model.fit(X_scaled, y)
joblib.dump(best_model, '../models/carbon_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(label_encoders, '../models/label_encoders.pkl')
print('Saved model artifacts successfully.')